# 00 · Setup & data
Replication of **"Real-Time Detection of Forest Fires Using FireNet-CNN and Explainable AI Techniques"** (Alam et al., IEEE Access 2025).

Notebook order: **00 setup/data → 01 synthetic + segmentation → 02 train FireNet-CNN → 03 baselines + ablation → 04 XAI + efficiency → 05 results vs. paper**.

Use a GPU runtime (*Runtime → Change runtime type → T4 GPU*). This notebook:
1. creates the project folder on Google Drive and writes the shared library `firenet.py` (+ the Stable-Diffusion script),
2. downloads / organises the three datasets into

```
firenet/data/A/{fire,nofire}        # Dataset A  - Shamta & Demir "Forest Fire Dataset" (Mendeley fcsjwd9gr6)
firenet/data/B/{fire,nofire}        # Dataset B  - Kizilkaya et al. "Effective Forest Fire Detection" (Mendeley g5nzp6j3bt)
firenet/data/synthetic/{fire,nofire}# Stable-Diffusion images of the authors (Mendeley nwzcm9ckrt)
```

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/firenet"
for d in ("data", "raw", "runs", "cache"):
    os.makedirs(f"{PROJECT}/{d}", exist_ok=True)
!nvidia-smi -L

## 1 · Write the shared library to Drive
This cell contains the whole implementation (model, CV loop, metrics, plots, Grad-CAM, saliency, efficiency). Other notebooks just `import firenet`.

In [ ]:
%%writefile /content/drive/MyDrive/firenet/firenet.py
"""
Replication of: "Real-Time Detection of Forest Fires Using FireNet-CNN and Explainable AI Techniques"
(Alam et al., IEEE Access 2025).

Expected data layout (class folders hold the images):
    data/A/fire, data/A/nofire            # Dataset A  (real)
    data/B/fire, data/B/nofire            # Dataset B  (real)
    data/synthetic/fire, data/synthetic/nofire   # Stable Diffusion images (generate_synthetic.py or Mendeley)

Sub-commands:
    segment     HSV + Canny fire-edge visualisation (Eq. 1-5)
    train       FireNet-CNN, 5-fold CV, metrics + curves + confusion matrices + ROC
    baselines   VGG16 / VGG19 / InceptionV3 fine-tuned under the same protocol
    ablation    Table 5 variants
    efficiency  params / model size / latency / GFLOPs (Table 4 & 7)
    xai         Grad-CAM + saliency maps for a trained model

Examples:
    python firenet.py train --dataset B --mode paper
    python firenet.py train --dataset B --mode strict          # synthetic images only in TRAIN folds
    python firenet.py baselines --dataset A
    python firenet.py xai --model runs/B_paper_firenet/model_fold1.keras --images a.jpg b.jpg
"""
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")   # Keras 2 behaviour on new TF (needs `pip install tf_keras`)
import argparse, gc, json, tempfile, time
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (confusion_matrix, matthews_corrcoef, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras import applications, layers, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG = 150
SHAPE = (IMG, IMG, 3)
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
GRADCAM_LAYER = "act5"   # ReLU output of the last conv block


# ----------------------------------------------------------------------------- data
def _read_folder(folder, label, synth, X, y, s):
    for p in sorted(Path(folder).glob("*")):
        if p.suffix.lower() not in EXT:
            continue
        im = cv2.imread(str(p))
        if im is None:
            continue
        im = cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2RGB), (IMG, IMG), interpolation=cv2.INTER_AREA)
        X.append(im); y.append(label); s.append(synth)


def load_dataset(root, name, use_synth=True):
    """label 1 = fire, 0 = no fire. Returns uint8 images, labels, is_synthetic flag."""
    X, y, s = [], [], []
    for label, cls in enumerate(["nofire", "fire"]):
        _read_folder(Path(root) / name / cls, label, False, X, y, s)
        if use_synth:
            _read_folder(Path(root) / "synthetic" / cls, label, True, X, y, s)
    X, y, s = np.stack(X), np.array(y), np.array(s)
    print(f"[data] {name}: {len(y)} imgs | fire={int(y.sum())} nofire={int((1-y).sum())} | synthetic={int(s.sum())}")
    return X, y, s


def load_cached(root, name, use_synth=True, cache_dir=None):
    """Like load_dataset, but caches the decoded uint8 arrays as .npz (big speed-up when images live on Google Drive)."""
    if cache_dir is None:
        return load_dataset(root, name, use_synth)
    f = Path(cache_dir) / f"{name}_{'syn' if use_synth else 'real'}_{IMG}.npz"
    if f.exists():
        d = np.load(f); print(f"[data] loaded cache {f}")
        return d["X"], d["y"], d["s"]
    X, y, s = load_dataset(root, name, use_synth)
    f.parent.mkdir(parents=True, exist_ok=True); np.savez_compressed(f, X=X, y=y, s=s)
    return X, y, s


def make_folds(y, synth, mode, seed, k=5):
    """
    mode='paper'  : synthetic images are pooled with real ones BEFORE splitting (what the paper describes;
                    synthetic images - and near-duplicate video frames - can land in validation folds).
    mode='strict' : folds are built from REAL images only; all synthetic images go to the training side.
    """
    skf = StratifiedKFold(k, shuffle=True, random_state=seed)
    if mode == "paper":
        yield from skf.split(np.zeros(len(y)), y)
    else:
        real, syn = np.where(~synth)[0], np.where(synth)[0]
        for tr, va in skf.split(real, y[real]):
            yield np.concatenate([real[tr], syn]), real[va]


# ----------------------------------------------------------------------------- segmentation (Eq. 1-5)
def segment_fire(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    m = cv2.inRange(hsv, np.array([0, 100, 100]), np.array([20, 255, 255]))
    b = cv2.GaussianBlur(m, (5, 5), 0)
    mc = cv2.morphologyEx(b, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    e = cv2.Canny(mc, 100, 200)
    ec = np.zeros_like(img_rgb); ec[e > 0] = (0, 255, 0)           # green edges
    return cv2.addWeighted(img_rgb, 1, ec, 0.5, 0)


def cmd_segment(a):
    X, y, _ = load_dataset(a.data, a.dataset, use_synth=False)
    idx = np.where(y == 1)[0][: a.n]
    fig, ax = plt.subplots(2, len(idx), figsize=(3 * len(idx), 6))
    for j, i in enumerate(idx):
        ax[0, j].imshow(X[i]); ax[0, j].set_title("Original Image", fontsize=8)
        ax[1, j].imshow(segment_fire(X[i])); ax[1, j].set_title("Detected Fire Edges", fontsize=8)
        ax[0, j].axis("off"); ax[1, j].axis("off")
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    plt.tight_layout(); plt.savefig(out / f"segmentation_{a.dataset}.png", dpi=150); plt.close()


# ----------------------------------------------------------------------------- models
def _act(x, act, name=None):
    return layers.LeakyReLU(name=name)(x) if act == "leaky_relu" else layers.Activation(act, name=name)(x)


def build_firenet(conv=True, act="relu", dropout=True, bn=True):
    """
    5 x [Conv3x3('valid') -> act -> MaxPool2x2 -> BN] with 32/64/128/256/512 filters,
    Flatten -> Dense512 -> Dropout.5 -> Dense256 -> Dropout.5 -> Dense1(sigmoid).
    With 150x150 input and 'valid' convs the last feature map is 2x2x512 and the model has exactly
    2,753,217 parameters - the number reported in the paper (Table 4).
    """
    inp = layers.Input(SHAPE)
    x = inp
    if conv:
        for i, f in enumerate([32, 64, 128, 256, 512], 1):
            x = layers.Conv2D(f, 3, padding="valid", name=f"conv{i}")(x)
            x = _act(x, act, name=f"act{i}")
            x = layers.MaxPooling2D(2, name=f"pool{i}")(x)
            if bn:
                x = layers.BatchNormalization(name=f"bn{i}")(x)
    x = layers.Flatten()(x)
    for u in (512, 256):
        x = layers.Dense(u)(x)
        x = _act(x, act)
        if dropout:
            x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inp, out, name="FireNet_CNN")


def build_pretrained(name):
    base = {"vgg16": applications.VGG16, "vgg19": applications.VGG19,
            "inceptionv3": applications.InceptionV3}[name](weights="imagenet", include_top=False, input_shape=SHAPE)
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(base.input, out, name=name)


def compile_model(m, opt="adam", loss="bce", lr=1e-4):
    o = {"adam": optimizers.Adam(lr), "rmsprop": optimizers.RMSprop(lr),
         "sgd": optimizers.SGD(1e-3, momentum=0.9)}[opt]
    m.compile(o, loss={"bce": "binary_crossentropy", "mse": "mean_squared_error"}[loss], metrics=["accuracy"])
    return m


# ----------------------------------------------------------------------------- metrics
def compute_metrics(y, p, thr=0.5):
    yh = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(accuracy=(tp + tn) / len(y), precision=prec, recall=rec,
                f1=2 * prec * rec / max(prec + rec, 1e-12), jaccard=tp / max(tp + fp + fn, 1),
                mcc=matthews_corrcoef(y, yh), auc=roc_auc_score(y, p), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))


# ----------------------------------------------------------------------------- plots
def plot_histories(hists, out, tag):
    for key, ttl in (("accuracy", "Accuracy"), ("loss", "Loss")):
        fig, axs = plt.subplots(1, len(hists), figsize=(4 * len(hists), 3.2))
        axs = np.atleast_1d(axs)
        for i, h in enumerate(hists):
            axs[i].plot(h[key], "b", label=f"Training {ttl}"); axs[i].plot(h["val_" + key], "r", label=f"Validation {ttl}")
            axs[i].set_title(f"Fold {i+1} - {ttl} Curve"); axs[i].set_xlabel("Epochs"); axs[i].grid(alpha=.3); axs[i].legend(fontsize=7)
        plt.tight_layout(); plt.savefig(out / f"{tag}_{key}_per_fold.png", dpi=130); plt.close()
        plt.figure(figsize=(8, 5))
        for i, h in enumerate(hists):
            c = f"C{i}"; plt.plot(h[key], "--", c=c, label=f"Training {ttl} - Fold {i+1}"); plt.plot(h["val_" + key], c=c, label=f"Validation {ttl} - Fold {i+1}")
        plt.title(f"Training and Validation {ttl} Curves for All Folds"); plt.xlabel("Epochs"); plt.legend(fontsize=6); plt.grid(alpha=.3)
        plt.savefig(out / f"{tag}_{key}_all_folds.png", dpi=130); plt.close()


def plot_confusions(cms, out, tag):
    fig, axs = plt.subplots(1, len(cms) + 1, figsize=(3.4 * (len(cms) + 1), 3.2))
    for ax, cm, t in zip(axs, cms + [np.mean(cms, 0).round().astype(int)], [f"Fold {i+1}" for i in range(len(cms))] + ["Mean"]):
        ax.imshow(cm, cmap="YlGn")
        for (i, j), v in np.ndenumerate(cm): ax.text(j, i, int(v), ha="center", va="center")
        ax.set_xticks([0, 1], ["No Fire", "Fire"]); ax.set_yticks([0, 1], ["No Fire", "Fire"]); ax.set_title(f"Confusion Matrix - {t}", fontsize=8)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout(); plt.savefig(out / f"{tag}_confusion.png", dpi=130); plt.close()


def plot_roc(roc_data, aucs, out, tag):
    plt.figure(figsize=(5, 4.5)); grid = np.linspace(0, 1, 200); tprs = []
    for i, (fpr, tpr) in enumerate(roc_data):
        plt.plot(fpr, tpr, lw=1, label=f"ROC Fold {i+1} (AUC = {aucs[i]:.2f})"); tprs.append(np.interp(grid, fpr, tpr))
    plt.plot(grid, np.mean(tprs, 0), "b--", label=f"Mean ROC (AUC = {np.mean(aucs):.4f})"); plt.plot([0, 1], [0, 1], "r--", label="Random Chance")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.legend(fontsize=7); plt.grid(alpha=.3)
    plt.savefig(out / f"{tag}_roc.png", dpi=130); plt.close()


# ----------------------------------------------------------------------------- CV loop
def run_cv(X, y, synth, model_fn, args, tag, compile_kw=None):
    compile_kw = compile_kw or {}
    out = Path(args.out) / f"{args.dataset}_{args.mode}_{tag}"; out.mkdir(parents=True, exist_ok=True)
    train_gen = ImageDataGenerator(rescale=1 / 255., rotation_range=20, width_shift_range=.1, height_shift_range=.1,
                                   shear_range=.1, zoom_range=.2, horizontal_flip=True, brightness_range=(.8, 1.2), fill_mode="nearest")
    val_gen = ImageDataGenerator(rescale=1 / 255.)
    rows, hists, cms, rocs = [], [], [], []
    for k, (tr, va) in enumerate(make_folds(y, synth, args.mode, args.seed), 1):
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(args.seed + k)
        m = compile_model(model_fn(), lr=args.lr, **compile_kw)
        t0 = time.time()
        h = m.fit(train_gen.flow(X[tr].astype("float32"), y[tr], batch_size=args.bs, shuffle=True, seed=args.seed),
                  validation_data=val_gen.flow(X[va].astype("float32"), y[va], batch_size=args.bs, shuffle=False),
                  epochs=args.epochs, verbose=2).history
        p = m.predict(val_gen.flow(X[va].astype("float32"), batch_size=64, shuffle=False), verbose=0).ravel()
        r = compute_metrics(y[va], p); r.update(fold=k, train_time_s=time.time() - t0, train_acc=h["accuracy"][-1],
                                                val_loss=h["val_loss"][-1], train_loss=h["loss"][-1], n_train=len(tr), n_val=len(va))
        rows.append(r); hists.append(h); cms.append(np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]])); rocs.append(roc_curve(y[va], p)[:2])
        print(f"[{tag}] fold {k}: " + " ".join(f"{q}={r[q]:.4f}" for q in ("accuracy", "precision", "recall", "f1", "mcc", "auc")), flush=True)
        if k == 1:
            m.save(out / "model_fold1.keras")
        del m; gc.collect()          # free the float32 copies of the folds before the next one
    df = pd.DataFrame(rows); df.to_csv(out / "fold_results.csv", index=False)
    json.dump(hists, open(out / "histories.json", "w"))
    plot_histories(hists, out, tag); plot_confusions(cms, out, tag); plot_roc(rocs, df["auc"].values, out, tag)
    summ = df[["train_acc", "accuracy", "precision", "recall", "f1", "jaccard", "mcc", "auc", "train_loss", "val_loss", "train_time_s"]].mean()
    print(f"\n=== {tag} | {args.dataset} | mode={args.mode} | mean over {len(rows)} folds ===\n{summ.round(4).to_string()}\n")
    summ.to_json(out / "summary.json")
    return summ


def _load(args):
    return load_dataset(args.data, args.dataset, use_synth=not args.no_synth)


def cmd_train(a):
    X, y, s = _load(a); run_cv(X, y, s, build_firenet, a, "firenet")


def cmd_baselines(a):
    X, y, s = _load(a); res = {}
    for n in ("vgg16", "vgg19", "inceptionv3"):
        res[n] = run_cv(X, y, s, lambda n=n: build_pretrained(n), a, n)
    pd.DataFrame(res).T.to_csv(Path(a.out) / f"{a.dataset}_{a.mode}_baselines.csv")


def cmd_ablation(a):
    X, y, s = _load(a)
    V = {"original": ({}, {}), "no_conv": ({"conv": False}, {}), "leakyrelu": ({"act": "leaky_relu"}, {}),
         "sigmoid_act": ({"act": "sigmoid"}, {}), "no_dropout": ({"dropout": False}, {}), "no_bn": ({"bn": False}, {}),
         "sgd": ({}, {"opt": "sgd"}), "rmsprop": ({}, {"opt": "rmsprop"}), "mse_loss": ({}, {"loss": "mse"})}
    res = {n: run_cv(X, y, s, lambda b=b: build_firenet(**b), a, f"abl_{n}", c) for n, (b, c) in V.items()}
    df = pd.DataFrame(res).T[["train_acc", "accuracy", "precision", "f1"]] * 100
    df.columns = ["Train Acc (%)", "Val Acc (%)", "Precision (%)", "F1 (%)"]
    df.round(2).to_csv(Path(a.out) / f"{a.dataset}_{a.mode}_ablation.csv"); print(df.round(2))


# ----------------------------------------------------------------------------- efficiency
def get_gflops(model):
    try:
        from tensorflow.python.framework import convert_to_constants as _c
        cv = getattr(_c, "convert_variables_to_constants_v2")
        cf = tf.function(lambda x: model(x)).get_concrete_function(tf.TensorSpec([1, *SHAPE], tf.float32))
        gd = cv(cf).graph.as_graph_def()          # convert in eager mode, profile in a fresh graph
        with tf.Graph().as_default() as g:
            tf.graph_util.import_graph_def(gd, name="")
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation(); opts["output"] = "none"
            info = tf.compat.v1.profiler.profile(g, options=opts)
        return info.total_float_ops / 1e9
    except Exception as e:
        print("FLOP profiling failed:", e); return float("nan")


def cmd_efficiency(a):
    rows = []
    for n, fn in (("FireNet-CNN", build_firenet), ("VGG16", lambda: build_pretrained("vgg16")),
                  ("VGG19", lambda: build_pretrained("vgg19")), ("InceptionV3", lambda: build_pretrained("inceptionv3"))):
        tf.keras.backend.clear_session(); m = fn()
        with tempfile.TemporaryDirectory() as d:
            m.save(Path(d) / "m.keras"); size = os.path.getsize(Path(d) / "m.keras") / 2**20
        x1 = np.random.rand(1, *SHAPE).astype("float32"); xb = np.random.rand(32, *SHAPE).astype("float32")
        for _ in range(5): m.predict(x1, verbose=0)
        t = []
        for _ in range(50):
            t0 = time.perf_counter(); m.predict(x1, verbose=0); t.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); m.predict(xb, verbose=0); tb = (time.perf_counter() - t0) / 32
        rows.append(dict(model=n, params=m.count_params(), size_MB=round(size, 2), latency_ms_per_img_bs1=round(1e3 * np.median(t), 1),
                         ms_per_img_bs32=round(1e3 * tb, 2), GFLOPs=round(get_gflops(m), 3)))
    df = pd.DataFrame(rows); print(df.to_string(index=False)); Path(a.out).mkdir(parents=True, exist_ok=True); df.to_csv(Path(a.out) / "efficiency.csv", index=False)


# ----------------------------------------------------------------------------- XAI
def gradcam(model, x, layer=GRADCAM_LAYER):
    gm = tf.keras.Model(model.inputs, [model.get_layer(layer).output, model.output])
    with tf.GradientTape() as tape:
        conv, pred = gm(x[None]); score = pred[:, 0]          # P(fire)
    g = tape.gradient(score, conv); w = tf.reduce_mean(g, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1))[0].numpy()
    return cv2.resize(cam / (cam.max() + 1e-8), (x.shape[1], x.shape[0])), float(pred[0, 0])


def saliency(model, x):
    xt = tf.convert_to_tensor(x[None])
    with tf.GradientTape() as tape:
        tape.watch(xt); score = model(xt)[:, 0]
    s = tf.reduce_max(tf.abs(tape.gradient(score, xt)), axis=-1)[0].numpy()
    return (s - s.min()) / (s.max() - s.min() + 1e-8)


def cmd_xai(a):
    m = tf.keras.models.load_model(a.model); out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(len(a.images), 3, figsize=(9, 3 * len(a.images))); ax = np.atleast_2d(ax)
    for i, pth in enumerate(a.images):
        im = cv2.cvtColor(cv2.resize(cv2.imread(pth), (IMG, IMG)), cv2.COLOR_BGR2RGB); x = im.astype("float32") / 255.
        cam, p = gradcam(m, x); sal = saliency(m, x)
        heat = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype("uint8"), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        ax[i, 0].imshow(im); ax[i, 0].set_title(f"Original (P(fire)={p:.3f})", fontsize=8)
        ax[i, 1].imshow(cv2.addWeighted(im, .6, heat, .4, 0)); ax[i, 1].set_title("Grad-CAM", fontsize=8)
        ax[i, 2].imshow(sal, cmap="hot"); ax[i, 2].set_title("Saliency map", fontsize=8)
        for q in ax[i]: q.axis("off")
    plt.tight_layout(); plt.savefig(out / "xai.png", dpi=150); plt.close(); print("saved", out / "xai.png")


# ----------------------------------------------------------------------------- CLI
def main():
    ap = argparse.ArgumentParser(); sub = ap.add_subparsers(dest="cmd", required=True)
    def common(p):
        p.add_argument("--data", default="data"); p.add_argument("--dataset", choices=["A", "B"], default="B")
        p.add_argument("--mode", choices=["paper", "strict"], default="paper"); p.add_argument("--no_synth", action="store_true")
        p.add_argument("--epochs", type=int, default=50); p.add_argument("--bs", type=int, default=32)
        p.add_argument("--lr", type=float, default=1e-4); p.add_argument("--seed", type=int, default=42); p.add_argument("--out", default="runs")
        return p
    for n, f in (("train", cmd_train), ("baselines", cmd_baselines), ("ablation", cmd_ablation)):
        common(sub.add_parser(n)).set_defaults(fn=f)
    p = sub.add_parser("segment"); p.add_argument("--data", default="data"); p.add_argument("--dataset", default="A"); p.add_argument("--n", type=int, default=5); p.add_argument("--out", default="runs"); p.set_defaults(fn=cmd_segment)
    p = sub.add_parser("efficiency"); p.add_argument("--out", default="runs"); p.set_defaults(fn=cmd_efficiency)
    p = sub.add_parser("xai"); p.add_argument("--model", required=True); p.add_argument("--images", nargs="+", required=True); p.add_argument("--out", default="runs"); p.set_defaults(fn=cmd_xai)
    a = ap.parse_args(); a.fn(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/firenet/generate_synthetic.py
"""
Generate synthetic fire / no-fire forest images with Stable Diffusion (paper Sec. III-B).

Paper: 1,020 fire + 1,181 non-fire images, prompts "a fire in a forest" / "a forest without fire",
CLIP ViT-B/32 similarity (CLIP score) between 35.56 and 45.23.

NOTE: you can SKIP this script and download the authors' generated images from Mendeley Data
(https://data.mendeley.com/datasets/nwzcm9ckrt/1) into data/synthetic/{fire,nofire}.

Usage:
    python generate_synthetic.py --out data/synthetic --n_fire 1020 --n_nofire 1181 --clip_score
"""
import argparse, random
from pathlib import Path
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

BASE = {"fire": "a fire in a forest", "nofire": "a forest without fire"}
# Optional modifiers -> more diversity than the 2 bare prompts (paper only lists the bare prompts)
MODS = ["", ", daytime", ", at night", ", dense smoke", ", aerial view", ", autumn", ", misty morning",
        ", high resolution photo", ", wide angle", ", close up"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/synthetic")
    ap.add_argument("--n_fire", type=int, default=1020)
    ap.add_argument("--n_nofire", type=int, default=1181)
    ap.add_argument("--model", default="stable-diffusion-v1-5/stable-diffusion-v1-5")
    ap.add_argument("--steps", type=int, default=50)       # paper says 1000 train steps; 50 is the usual sampling budget
    ap.add_argument("--bs", type=int, default=4)
    ap.add_argument("--diverse", action="store_true", help="append random modifiers to prompts")
    ap.add_argument("--clip_score", action="store_true")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    pipe = StableDiffusionPipeline.from_pretrained(
        a.model, torch_dtype=torch.float16 if dev == "cuda" else torch.float32, safety_checker=None).to(dev)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

    clip = None
    if a.clip_score:
        from torchmetrics.multimodal.clip_score import CLIPScore
        clip = CLIPScore(model_name_or_path="openai/clip-vit-base-patch32").to(dev)

    rng = random.Random(a.seed)
    gen = torch.Generator(dev).manual_seed(a.seed)
    scores = []
    for cls, n in (("fire", a.n_fire), ("nofire", a.n_nofire)):
        d = Path(a.out) / cls
        d.mkdir(parents=True, exist_ok=True)
        done = len(list(d.glob("*.png")))
        while done < n:
            k = min(a.bs, n - done)
            prompts = [BASE[cls] + (rng.choice(MODS) if a.diverse else "") for _ in range(k)]
            imgs = pipe(prompts, num_inference_steps=a.steps, generator=gen).images
            for p, im in zip(prompts, imgs):
                im.save(d / f"sd_{cls}_{done:05d}.png")
                if clip is not None:
                    import numpy as np
                    t = torch.from_numpy(np.array(im)).permute(2, 0, 1).to(dev)
                    scores.append(float(clip(t, p)))
                done += 1
            print(f"{cls}: {done}/{n}", flush=True)
    if scores:
        print(f"CLIP score  min={min(scores):.2f}  max={max(scores):.2f}  mean={sum(scores)/len(scores):.2f}")


if __name__ == "__main__":
    main()

## 2 · Get the raw data
**Option A - manual (most reliable).** Open each page, click *Download All*, and upload the zips to `MyDrive/firenet/raw/` named exactly `A.zip`, `B.zip`, `synthetic.zip`:

| name | page | version |
|---|---|---|
| `A.zip` | https://data.mendeley.com/datasets/fcsjwd9gr6 | 2 |
| `B.zip` | https://data.mendeley.com/datasets/g5nzp6j3bt | 3 (paper doesn't state; ~3,400 images expected) |
| `synthetic.zip` | https://data.mendeley.com/datasets/nwzcm9ckrt | 1 |

**Option B - try the Mendeley zip endpoint** (experimental; I could not test it from my sandbox). If it returns an HTML page or an error, use Option A.

In [ ]:
import requests, zipfile, io
def mendeley_zip(ds_id, version, dest):
    url = f"https://data.mendeley.com/public-api/zip/{ds_id}/download/{version}"
    r = requests.get(url, stream=True, timeout=300); r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(1 << 20): f.write(chunk)
    print(dest, os.path.getsize(dest) // 2**20, "MB", "| valid zip:", zipfile.is_zipfile(dest))

RAW = f"{PROJECT}/raw"
TRY_API = False        # set True to attempt Option B
if TRY_API:
    for name, ds, ver in (("A", "fcsjwd9gr6", 2), ("B", "g5nzp6j3bt", 3), ("synthetic", "nwzcm9ckrt", 1)):
        if not os.path.exists(f"{RAW}/{name}.zip"):
            try: mendeley_zip(ds, ver, f"{RAW}/{name}.zip")
            except Exception as e: print(name, "failed ->", e, "-> use Option A")

In [ ]:
for name in ("A", "B", "synthetic"):
    z = f"{RAW}/{name}.zip"
    if os.path.exists(z) and zipfile.is_zipfile(z):
        zipfile.ZipFile(z).extractall(f"/content/raw_{name}")            # extract to local disk (fast)
        print("extracted", name)
    else:
        print("MISSING / invalid:", z)
# Mendeley zips sometimes contain nested zips - extract those too
import glob
for z in glob.glob("/content/raw_*/**/*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(os.path.dirname(z)); print("nested:", z)
!for n in A B synthetic; do echo "== $n"; find /content/raw_$n -maxdepth 3 -type d | head -30; done

## 3 · Organise into `fire/` and `nofire/`
The folder names inside the Mendeley zips are not known in advance, so this cell **infers the class from each image's immediate parent folder** and prints an audit table. **Read the table.** Anything ambiguous is skipped and listed under `UNKNOWN`; fix it with `FORCE` (folder → 0/1) or `EXCLUDE` (substring of the path to drop, e.g. the object-detection part of Dataset A).

In [ ]:
import re, shutil, hashlib
from pathlib import Path
from collections import Counter
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
NOFIRE = re.compile(r"(no[\s_\-]?fire|non[\s_\-]?fire|not[\s_\-]?fire|without[\s_\-]?fire|fire[\s_\-]?free)", re.I)
NOFIRE_WORDS = {"forest", "forests", "normal", "intact", "nature", "negative", "neutral", "no", "non"}

def infer(folder_name):
    n = folder_name.lower()
    if NOFIRE.search(n) or n in NOFIRE_WORDS: return 0
    if "fire" in n: return 1
    return None

def organize(raw_root, dst, exclude=(), force=None, dry=True):
    force = force or {}
    raw_root, dst = Path(raw_root), Path(dst)
    audit, plan = Counter(), []
    for p in raw_root.rglob("*"):
        if p.suffix.lower() not in EXT: continue
        rel = p.relative_to(raw_root); rels = str(rel.parent)
        if any(k.lower() in str(rel).lower() for k in exclude): audit[(rels, "EXCLUDED")] += 1; continue
        lab = force.get(rels, infer(p.parent.name))
        audit[(rels, {0: "nofire", 1: "fire", None: "UNKNOWN"}[lab])] += 1
        if lab is not None: plan.append((p, lab, rel))
    print(pd.Series(audit).rename("n_images").to_string() if audit else "no images found")
    c = Counter(l for _, l, _ in plan); print(f"\n-> fire={c[1]}  nofire={c[0]}  total={len(plan)}")
    if not dry:
        for cls in ("fire", "nofire"): (dst / cls).mkdir(parents=True, exist_ok=True)
        for p, lab, rel in plan:
            h = hashlib.md5(str(rel).encode()).hexdigest()[:8]
            shutil.copy(p, dst / ("fire" if lab else "nofire") / f"{h}_{p.name}")
        print("copied to", dst)
import pandas as pd

In [ ]:
# ---- DATASET A (expected total: 2,974 classification images; exclude the object-detection part if it shows up) ----
EXCLUDE_A = ["detect", "yolo", "object", "annot", "label"]
organize("/content/raw_A", f"{PROJECT}/data/A", exclude=EXCLUDE_A, dry=True)

In [ ]:
# ---- DATASET B (expected ~3,400: ~1,111 fire / ~2,289 non-fire) ----
organize("/content/raw_B", f"{PROJECT}/data/B", dry=True)

In [ ]:
# ---- SYNTHETIC (expected 2,204: 1,102 + 1,102) ----
organize("/content/raw_synthetic", f"{PROJECT}/data/synthetic", dry=True)

If every audit table looks right, set `dry=False` below (add `force={...}` / `exclude=[...]` from above if you needed them). It copies the files to Drive.

In [ ]:
organize("/content/raw_A", f"{PROJECT}/data/A", exclude=EXCLUDE_A, dry=False)
organize("/content/raw_B", f"{PROJECT}/data/B", dry=False)
organize("/content/raw_synthetic", f"{PROJECT}/data/synthetic", dry=False)

In [ ]:
# final sanity check
for d in ("A", "B", "synthetic"):
    for c in ("fire", "nofire"):
        print(f"{d:10s} {c:7s}", len([f for f in os.listdir(f"{PROJECT}/data/{d}/{c}")]))